In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ribonanzanet2/pytorch/alpha/1/dropout.py
/kaggle/input/ribonanzanet2/pytorch/alpha/1/pairwise.yaml
/kaggle/input/ribonanzanet2/pytorch/alpha/1/Network.py
/kaggle/input/ribonanzanet2/pytorch/alpha/1/pytorch_model_fsdp.bin
/kaggle/input/stanford-rna-3d-folding/sample_submission.csv
/kaggle/input/stanford-rna-3d-folding/validation_sequences.csv
/kaggle/input/stanford-rna-3d-folding/test_sequences.csv
/kaggle/input/stanford-rna-3d-folding/train_labels.v2.csv
/kaggle/input/stanford-rna-3d-folding/validation_labels.csv
/kaggle/input/stanford-rna-3d-folding/train_labels.csv
/kaggle/input/stanford-rna-3d-folding/train_sequences.csv
/kaggle/input/stanford-rna-3d-folding/train_sequences.v2.csv
/kaggle/input/stanford-rna-3d-folding/MSA/R1108.MSA.fasta
/kaggle/input/stanford-rna-3d-folding/MSA/8EVR_EC.MSA.fasta
/kaggle/input/stanford-rna-3d-folding/MSA/1ZDI_S.MSA.fasta
/kaggle/input/stanford-rna-3d-folding/MSA/5FJ1_H.MSA.fasta
/kaggle/input/stanford-rna-3d-folding/MSA/2NBY_A.MSA.fast

In [2]:
import kagglehub

# Download latest version
path = kagglehub.model_download("shujun717/ribonanzanet2/pyTorch/alpha")

print("Path to model files:", path)

Path to model files: /kaggle/input/ribonanzanet2/pytorch/alpha/1


In [3]:
import pandas as pd
import torch
import os
import yaml
import sys
sys.path.append('/kaggle/input/ribonanzanet2/pytorch/alpha/1')

# Cargar configuración del modelo
with open('/kaggle/input/ribonanzanet2/pytorch/alpha/1/pairwise.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Importar la clase del modelo desde Network.py
from Network import RibonanzaNet

# Ajustar el diccionario para que el modelo lo acepte (si no usa corchetes)
# Esto es opcional si ya corregiste Network.py
class ConfigWrapper:
    def __init__(self, config_dict):
        self.__dict__.update(config_dict)

config_obj = ConfigWrapper(config)

# Inicializar el modelo con la configuración
model = RibonanzaNet(config_obj)

# Cargar pesos desde el archivo .bin
model_weights_path = '/kaggle/input/ribonanzanet2/pytorch/alpha/1/pytorch_model_fsdp.bin'
model.load_state_dict(torch.load(model_weights_path, map_location='cpu'))
model.eval()

constructing 48 ConvTransformerEncoderLayers


/tmp/ipykernel_13/668963045.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_weights_path, map_location='cpu'))


RibonanzaNet(
  (transformer_encoder): ModuleList(
    (0-47): 48 x ConvTransformerEncoderLayer(
      (self_attn): MultiHeadAttention(
        (w_qs): Linear(in_features=384, out_features=384, bias=False)
        (w_ks): Linear(in_features=384, out_features=384, bias=False)
        (w_vs): Linear(in_features=384, out_features=384, bias=False)
        (attention): ScaledDotProductAttention(
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (pairwise2heads): Linear(in_features=128, out_features=12, bias=False)
      (pairwise_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (activation): GELU(approximate='none')
      (triangle_update_out): TriangleMultiplicativeModule(
        (norm): LayerNorm((128,), eps=1e-05, 

In [4]:
import pandas as pd
import torch
import os
import yaml
import sys
sys.path.append('/kaggle/input/ribonanzanet2/pytorch/alpha/1')

# Cargar configuración del modelo
with open('/kaggle/input/ribonanzanet2/pytorch/alpha/1/pairwise.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Importar la clase del modelo desde Network.py
from Network import RibonanzaNet

# Ajustar el diccionario para que el modelo lo acepte
class ConfigWrapper:
    def __init__(self, config_dict):
        self.__dict__.update(config_dict)

config_obj = ConfigWrapper(config)

# Inicializar el modelo con la configuración
model = RibonanzaNet(config_obj)
model_weights_path = '/kaggle/input/ribonanzanet2/pytorch/alpha/1/pytorch_model_fsdp.bin'

# Cargar pesos con seguridad (weights_only=True)
model.load_state_dict(torch.load(model_weights_path, map_location='cpu', weights_only=True))
model.eval()

# Codificar secuencias como índices categóricos
def categorical_encode(sequence):
    nucleotides = {'A': 0, 'U': 1, 'C': 2, 'G': 3}
    encoded = [nucleotides[nt] for nt in sequence]
    return torch.tensor(encoded, dtype=torch.long)

# Cargar datos de prueba
test_df = pd.read_csv('/kaggle/input/stanford-rna-3d-folding/test_sequences.csv')

# Inicializar lista para almacenar resultados
submission_rows = []

# Generar predicciones para cada secuencia
for _, row in test_df.iterrows():
    target_id = row['target_id']
    sequence = row['sequence']
    
    # Codificar la secuencia
    encoded_seq = categorical_encode(sequence)
    encoded_seq = encoded_seq.unsqueeze(0)  # Shape: (1, L)
    
    # Crear src_mask
    src_mask = torch.ones_like(encoded_seq).float()  # Shape: (1, L)
    
    # Generar predicciones
    with torch.no_grad():
        coords = model(encoded_seq, src_mask=src_mask)  # Shape: (1, L, 10) o (1, L, 5, 3)
        coords = coords.squeeze(0)  # Shape: (L, 10) o (L, 5, 3)
        
        # Verificar y ajustar forma de salida
        if coords.dim() == 2 and coords.shape[1] == 10:
            # Asumimos que son 5 estructuras con 2 coordenadas (x, y)
            coords = coords.view(-1, 5, 2)  # Shape: (L, 5, 2)
            # Añadir un tercer valor cero para cumplir con el formato esperado (L, 5, 3)
            coords = torch.cat([coords, torch.zeros(coords.shape[0], coords.shape[1], 1)], dim=2)
        elif coords.dim() == 3 and coords.shape[1] == 5 and coords.shape[2] == 3:
            pass  # Forma correcta (L, 5, 3)
        else:
            raise ValueError(f"Forma de salida inesperada: {coords.shape}")

    # Mapear residuos y coordenadas
    for i, (resid, resname) in enumerate(zip(range(1, len(sequence)+1), sequence)):
        coords_list = coords[i].tolist()  # Shape: (5, 3)
        
        # Validar estructura de coords_list
        if any(not isinstance(coord, list) or len(coord) != 3 for coord in coords_list):
            raise ValueError(f"Estructura de coords_list inválida: {coords_list}")
        
        # Formatear coordenadas con 3 decimales
        coords_str = []
        for j in range(5):
            coords_str.extend([f"{coords_list[j][0]:.3f}", f"{coords_list[j][1]:.3f}", f"{coords_list[j][2]:.3f}"])
        
        # Crear fila para el archivo de envío
        submission_rows.append({
            'ID': f"{target_id}_{resid}",
            'resname': resname,
            'resid': resid,
            **{f'x_{j+1}': coords_str[3*j] for j in range(5)},
            **{f'y_{j+1}': coords_str[3*j+1] for j in range(5)},
            **{f'z_{j+1}': coords_str[3*j+2] for j in range(5)}
        })

# Crear DataFrame de envío
submission_df = pd.DataFrame(submission_rows)

# Guardar archivo de envío
submission_df.to_csv('submission.csv', index=False)

constructing 48 ConvTransformerEncoderLayers
